# Earth analogs for Mars: similarity table (Earthdata / MERRA-2)
Compares candidate Earth sites with reference Mars values on 4 variables: surface temperature, elevation, dust, vegetation (LAI).
**Before running:** create a free account at https://urs.earthdata.nasa.gov

In [ ]:
!pip install -q earthaccess xarray netCDF4 pandas matplotlib folium

## 1) Login to Earthdata (it will ask for your username and password)

In [ ]:
import earthaccess
auth = earthaccess.login()   # interactive login

## 2) Settings you must edit
- `SITES`: your 5 candidate sites (lat, lon). The ones below are only placeholders.
- `MARS`: reference values for Mars. **Replace them with numbers from a source you can cite** (NASA Mars Fact Sheet, papers, Mars Trek).

In [ ]:
YEAR = 2022   # year of Earth data to use

SITES = {
    "Lut Desert (Iran)":        (30.5, 59.0),
    "Gilf Kebir (Egypt)":       (23.3, 26.1),
    "Tibesti (Chad)":           (21.0, 17.5),
    "Puna de Atacama (Chile)":  (-25.0, -67.0),
    "Taklamakan (China)":       (39.0, 83.0),
}

# Reference Mars values (EDIT + CITE the source for each one)
MARS = {
    "ts_mean_K":   210,    # mean surface temperature, approx. 210 K
    "elevation_m": 0,      # use the elevation of your Mars target site (MOLA), 0 = datum
    "dust_aod":    0.5,    # approx. background dust optical depth, verify
    "lai":         0.0,    # no vegetation
}

# How much difference gives score 0 (the larger, the more forgiving)
SCALE   = {"ts_mean_K": 50, "elevation_m": 5000, "dust_aod": 0.5, "lai": 1.0}
WEIGHTS = {"ts_mean_K": 1, "elevation_m": 1, "dust_aod": 1, "lai": 1}

## 3) Download MERRA-2 monthly data (NetCDF) from Earthdata

In [ ]:
import xarray as xr

def get_files(short_name, **kw):
    res = earthaccess.search_data(short_name=short_name, **kw)
    print(short_name, "->", len(res), "granules")
    return earthaccess.download(res, "data")

period = dict(temporal=(f"{YEAR}-01-01", f"{YEAR}-12-31"))
slv_files = get_files("M2TMNXSLV", **period)   # temperature (TS), pressure
aer_files = get_files("M2TMNXAER", **period)   # aerosols / dust (DUEXTTAU)
lnd_files = get_files("M2TMNXLND", **period)   # land surface (LAI)
cnst_files = get_files("M2C0NXASM")            # constants (PHIS -> elevation)

slv  = xr.open_mfdataset(slv_files,  combine="by_coords")
aer  = xr.open_mfdataset(aer_files,  combine="by_coords")
lnd  = xr.open_mfdataset(lnd_files,  combine="by_coords")
cnst = xr.open_dataset(cnst_files[0])
print(list(slv.data_vars)[:15])   # if a variable name errors later, check names here

## 4) Extract values for each site

In [ ]:
import pandas as pd

def at(ds, var, lat, lon):
    return ds[var].sel(lat=lat, lon=lon, method="nearest")

rows = []
for name, (lat, lon) in SITES.items():
    ts   = at(slv, "TS", lat, lon).values            # K, one value per month
    dust = at(aer, "DUEXTTAU", lat, lon).values      # dust extinction AOD
    lai  = at(lnd, "LAI", lat, lon).values
    phis = at(cnst, "PHIS", lat, lon).values.squeeze()
    rows.append({
        "site": name, "lat": lat, "lon": lon,
        "ts_mean_K": float(ts.mean()),
        "ts_seasonal_range_K": float(ts.max() - ts.min()),
        "elevation_m": float(phis) / 9.80665,
        "dust_aod": float(dust.mean()),
        "lai": float(lai.mean()),
    })
df = pd.DataFrame(rows)
df

## 5) Similarity scores (0-100 per variable + composite)

In [ ]:
for v in SCALE:
    df["score_" + v] = (100 * (1 - (df[v] - MARS[v]).abs() / SCALE[v])).clip(lower=0)

w = pd.Series(WEIGHTS)
df["score_total"] = sum(df["score_" + v] * w[v] for v in SCALE) / w.sum()
df = df.sort_values("score_total", ascending=False).reset_index(drop=True)
df.round(2)

## 6) Chart of scores per variable (shows where each site is strong or weak)

In [ ]:
import matplotlib.pyplot as plt
cols = ["score_" + v for v in SCALE]
ax = df.set_index("site")[cols].plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Similarity to Mars (0-100)")
ax.set_title("Similarity per variable")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("scores.png", dpi=150)
plt.show()

## 7) Interactive map

In [ ]:
import folium

m = folium.Map(location=[20, 30], zoom_start=2, tiles="CartoDB positron")
for _, r in df.iterrows():
    popup = (f"<b>{r['site']}</b><br>Total score: {r['score_total']:.0f}<br>"
             f"Temp: {r['score_ts_mean_K']:.0f} | Elevation: {r['score_elevation_m']:.0f}<br>"
             f"Dust: {r['score_dust_aod']:.0f} | Vegetation: {r['score_lai']:.0f}")
    color = "green" if r["score_total"] >= 70 else "orange" if r["score_total"] >= 40 else "red"
    folium.Marker([r["lat"], r["lon"]], popup=folium.Popup(popup, max_width=250),
                  icon=folium.Icon(color=color)).add_to(m)
m.save("map.html")
m

## 8) Save results (then download them / upload to GitHub)

In [ ]:
df.to_csv("results.csv", index=False)
from google.colab import files
files.download("results.csv"); files.download("map.html"); files.download("scores.png")